In [73]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

from sklearn import svm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

from sklearn.model_selection import cross_val_score


import os
import sys

from configparser import ConfigParser
from pathlib import Path

# Set the working directory to the project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src" / "insurance_pricing").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

# Make your source package importable
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from Insurance_pricing.data import Find_N_datasets, inspect_dataset
from Insurance_pricing.cleaning import clean_types, clean_dates

In [48]:
print(os.getcwd())

c:\Users\alloh\OneDrive\Documents\Github\Motor-Insurance-Claims-Prediction-and-Pricing


In [49]:
paths = Find_N_datasets(2)
motor = pd.read_csv(paths[0], sep=";")
claims = pd.read_csv(paths[1], sep=";")

motor_data: C:\Users\alloh\OneDrive\Documents\Github\Motor-Insurance-Claims-Prediction-and-Pricing\Data\Raw\Motor.csv
claims_data: C:\Users\alloh\OneDrive\Documents\Github\Motor-Insurance-Claims-Prediction-and-Pricing\Data\Raw\Claims.csv


In [50]:
inspect_dataset(claims) #inspection shows there's no null and/or missing elements in the dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7366 entries, 0 to 7365
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   7366 non-null   int64  
 1   Cost_claims_year     7366 non-null   float64
 2   Cost_claims_by_type  7366 non-null   float64
 3   Claims_type          7366 non-null   object 
dtypes: float64(2), int64(1), object(1)
memory usage: 230.3+ KB

Info of the dataset:
None

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    

Checking the missing values:
ID                     0
Cost_claims_year       0
Cost_claims_by_type    0
Claims_type            0
dtype: int64

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
   

In [51]:
inspect_dataset(motor)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105555 entries, 0 to 105554
Data columns (total 30 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   ID                    105555 non-null  int64  
 1   Date_start_contract   105555 non-null  object 
 2   Date_last_renewal     105555 non-null  object 
 3   Date_next_renewal     105555 non-null  object 
 4   Date_birth            105555 non-null  object 
 5   Date_driving_licence  105555 non-null  object 
 6   Distribution_channel  105555 non-null  int64  
 7   Seniority             105555 non-null  int64  
 8   Policies_in_force     105555 non-null  int64  
 9   Max_policies          105555 non-null  int64  
 10  Max_products          105555 non-null  int64  
 11  Lapse                 105555 non-null  int64  
 12  Date_lapse            35147 non-null   object 
 13  Payment               105555 non-null  int64  
 14  Premium               105555 non-null  float64
 15  

In [52]:
motor = clean_dates(motor)

## Columns of interest

the columns of interest are:

Date_lapse, Type_fuel, Length

### Imprefection Qualities

all the values that're non-null are missing. And thus must mbe

In [53]:
missing_cols = motor[['Date_lapse', 'Type_fuel', 'Length']].copy()
missing_cols

,Date_lapse,Type_fuel,Length
0,NaT,P,NaN
1,NaT,P,NaN
2,NaT,P,NaN
3,NaT,P,NaN
4,NaT,P,NaN
...,...,...,...
105550,NaT,D,4.740
105551,NaT,P,4.650
105552,NaT,P,3.495
105553,NaT,D,4.555


In [54]:
missing_cols['Type_fuel'].unique()

array(['P', 'D', nan], dtype=object)

In [55]:
# gives the proportion of rows with missing values
missing_rows = missing_cols.isna().any(axis=1).sum()
missing_rows/len(missing_cols) * 100

np.float64(69.85647292880489)

Nearly 70% of rows have missing values. This isn't something that can be solved through simple removal of erroneous entries

In [56]:
inspect_dataset(missing_cols)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105555 entries, 0 to 105554
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   Date_lapse  35147 non-null   datetime64[ns]
 1   Type_fuel   103791 non-null  object        
 2   Length      95226 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 2.4+ MB

Info of the dataset:
None

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    

Checking the missing values:
Date_lapse    70408
Type_fuel      1764
Length        10329
dtype: int64

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
        
Stats for categorical columns:

       Type_fuel
count     103791
unique      

In [57]:
# Create a copy of the motor dataset
motor_check = motor.copy()

# Get the IDs present in the claims dataset
claim_ids = claims["ID"].dropna().astype("string").str.strip()

# Check whether each motor ID appears in the claims dataset
motor_ids = motor_check["ID"].astype("string").str.strip()
motor_check["Claim_Made"] = motor_ids.isin(claim_ids)

# Identify rows where Date_lapse is missing
missing_lapse = motor_check["Date_lapse"].isna()

# Missing Date_lapse where a claim exists
missing_with_claim = motor_check[
    missing_lapse & motor_check["Claim_Made"]
]

# Missing Date_lapse where no matching claim was found
missing_without_claim = motor_check[
    missing_lapse & ~motor_check["Claim_Made"]
]

# Summarise the results
print(f"Total rows: {len(motor_check):,}")
print(f"Rows missing Date_lapse: {missing_lapse.sum():,}")
print(f"Missing Date_lapse with a claim: {len(missing_with_claim):,}")
print(f"Missing Date_lapse without a matching claim: {len(missing_without_claim):,}")

# Count unique IDs as well
print(
    "Unique IDs missing Date_lapse with a claim:",
    missing_with_claim["ID"].nunique()
)

Total rows: 105,555
Rows missing Date_lapse: 70,408
Missing Date_lapse with a claim: 6,270
Missing Date_lapse without a matching claim: 64,138
Unique IDs missing Date_lapse with a claim: 2921


In [58]:
# 1 means there's a recorded termination date, 0 means no such date exists
motor_check["Contract_Terminated"] = (
    motor_check["Date_lapse"].notna().astype(int)
)

In [59]:
analysis = motor.copy()

for col in missing_cols:
    analysis[f"{col}_missing"] = analysis[col].isna().astype(int)

In [60]:
analysis.head()

,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,Power,Cylinder_capacity,Value_vehicle,N_doors,Type_fuel,Length,Weight,Date_lapse_missing,Type_fuel_missing,Length_missing
0,1,2015-11-05,2015-11-05,2016-11-05,1956-04-15,1976-03-20,0,4,1,2,...,80,599,7068.0,0,P,NaN,190,1,0,1
1,1,2015-11-05,2016-11-05,2017-11-05,1956-04-15,1976-03-20,0,4,1,2,...,80,599,7068.0,0,P,NaN,190,1,0,1
2,1,2015-11-05,2017-11-05,2018-11-05,1956-04-15,1976-03-20,0,4,2,2,...,80,599,7068.0,0,P,NaN,190,1,0,1
3,1,2015-11-05,2018-11-05,2019-11-05,1956-04-15,1976-03-20,0,4,2,2,...,80,599,7068.0,0,P,NaN,190,1,0,1
4,2,2017-09-26,2017-09-26,2018-09-26,1956-04-15,1976-03-20,0,4,2,2,...,80,599,7068.0,0,P,NaN,190,1,0,1


In [61]:
Length_imputation = motor[
    [
        'Type_risk',
        'Area',
        'Second_driver',
        'Power',
        'Cylinder_capacity',
        'Value_vehicle',
        'N_doors',
        'Length',
        'Weight',
        'Type_fuel'
    ]
].copy()
Length_imputation.dropna(subset=["Length","Type_fuel"], inplace=True)
Length_imputation = clean_types(Length_imputation)
Length_imputation.head()

,Type_risk,Area,Second_driver,Power,Cylinder_capacity,Value_vehicle,N_doors,Length,Weight,Type_fuel
6,3,0,0,85,1229,16030.0,5,3.999,1105,1
7,3,0,0,85,1229,16030.0,5,3.999,1105,1
8,3,0,0,85,1229,16030.0,5,3.999,1105,1
9,3,0,0,85,1229,16030.0,5,3.999,1105,1
15,3,0,0,87,1598,9927.0,4,4.250,1055,1


In [62]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

encoded = encoder.fit_transform(Length_imputation[["Type_risk"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Type_risk"]),
    index=Length_imputation.index
)

Length_imputation = pd.concat([Length_imputation, encoded_df], axis=1)
Length_imputation.drop(columns=["Type_risk"], inplace=True)
Length_imputation.head()

,Area,Second_driver,Power,Cylinder_capacity,Value_vehicle,N_doors,Length,Weight,Type_fuel,Type_risk_1,Type_risk_2,Type_risk_3,Type_risk_4
6,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
7,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
8,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
9,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
15,0,0,87,1598,9927.0,4,4.250,1055,1,0.0,0.0,1.0,0.0


In [63]:
Length_imputation.rename(columns={"Type_risk_1":"Is_Motorbike",
                                  "Type_risk_2":"Is_Van",
                                  "Type_risk_3":"Is_Passenger_Car",
                                  "Type_risk_4":"Is_Agricultural"},inplace=True)
Length_imputation.head()

,Area,Second_driver,Power,Cylinder_capacity,Value_vehicle,N_doors,Length,Weight,Type_fuel,Is_Motorbike,Is_Van,Is_Passenger_Car,Is_Agricultural
6,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
7,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
8,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
9,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
15,0,0,87,1598,9927.0,4,4.250,1055,1,0.0,0.0,1.0,0.0


In [64]:
Length_imputation.head()

,Area,Second_driver,Power,Cylinder_capacity,Value_vehicle,N_doors,Length,Weight,Type_fuel,Is_Motorbike,Is_Van,Is_Passenger_Car,Is_Agricultural
6,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
7,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
8,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
9,0,0,85,1229,16030.0,5,3.999,1105,1,0.0,0.0,1.0,0.0
15,0,0,87,1598,9927.0,4,4.250,1055,1,0.0,0.0,1.0,0.0


In [83]:
os.cpu_count()/2

8.0

In [ ]:
# Define the target and predictor columns
target = "Length"

features = [
    "Power",
    "Cylinder_capacity",
    "Is_Van",
    "Is_Passenger_Car",
    "Is_Agricultural",
    "Value_vehicle",
    "N_doors",
    "Weight",
    "Second_driver"
]

Learning_curve = []
# Define the target and predictors
target = "Length"

# Keep rows where Length is known
known = Length_imputation.dropna(subset=[target]).copy()

X = known[features]
y = known[target]


for i in range(0, 9):
    # Split into training and testing data
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size = 0.1 + float(f"0.{i}"),
        random_state=42
    )

    for j in range(5,200,5):
        # Build the RFR pipeline
        rfr = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=j,
                random_state=42,
                n_jobs= int(os.cpu_count() / 2)
            ))
        ])

        # Train and evaluate the model
        rfr.fit(X_train, y_train)

        predictions = rfr.predict(X_test)

        # Calculate permutation importance
        importance = permutation_importance(
            rfr,
            X_test,
            y_test,
            scoring="neg_mean_absolute_error",
            n_repeats=30,
            random_state=42,
            n_jobs=-1
        )
        
        # Display features in order of predictive importance
        importance_df = pd.DataFrame({
            "Feature": features,
            "Importance": importance.importances_mean,
            "Std": importance.importances_std
        }).sort_values("Importance", ascending=False)

        #Cross validation scoring
        scores = cross_val_score(rfr, X, y, cv=5) # we can just avoid the hyperparameter stuff for now...

        Learning_curve.append({
            "Train_size": len(X_train),
            "CV_scores": scores,
            "Mean_score": scores.mean(),
            "importance_dataframe": importance_df
        })


#print("Test MAE:", mean_absolute_error(y_test, predictions))

KeyboardInterrupt: 

In [ ]:
#Seniority,Policies_in_force,Max_policies,Max_products,Lapse,Payment,Premium,Cost_claims_year,Type_risk,Area